### TFT Analyzer Notebook

Notebook phân tích dữ liệu trên TFT bằng Spark. Nội dung chính:

Load & cache: đọc JSON từ thư mục, tạo lập data, cache, in schema.

Kiểm tra nhanh: sample thông tin trận, top 5 trận dài nhất.

Độ phổ biến tướng: bung units, top picks + biểu đồ cột.

Meta tướng: số trận + placement trung bình (càng thấp càng tốt); xem độ phổ biến của tướng mục tiêu (target_champ).

Meta tộc/hệ: trait được kích hoạt (tier_current > 0) theo num_units và placement trung bình.

Cách chạy:

Chạy tuần tự từng cell (Shift+Enter).

Nếu lỗi schema, kiểm tra đường dẫn ./data_3580_matches và cấu trúc file.

Có thể chỉnh cấu hình Spark (RAM/master) ở cell đầu.

Đổi target_champ để xem dữ liệu tướng khác.

Kết quả mong đợi:

Cây schema; top 5 trận dài; bảng top tướng; biểu đồ top 10 tướng.

Bảng meta tướng (số trận, placement trung bình); độ phổ biến của tướng mục tiêu.

Bảng meta tộc/hệ (tên, num_units, số trận, placement trung bình).


### Bước 1: Khởi tạo Spark và load dữ liệu

Tạo SparkSession, đọc JSON (từ thư mục, infer schema), unwrap trường data nếu có, cache kết quả.

Output: số trận đã load.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, count, desc, avg

spark = (
    SparkSession.builder
    .appName("TFT-Notebook")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(">>> Đang đọc dữ liệu...")
df = (
    spark.read.option("recursiveFileLookup", "true")
    .option("inferSchema", "true")
    .json("./data_3580_matches")
)
if "data" in df.columns and "info" not in df.columns:
    df = df.select("data.*")
df.cache()
print(f"Done! Đã load {df.count():,} trận đấu.")


In cây schema để nắm cấu trúc (metadata, info, participants, units/items/traits).

In [ ]:
print("=== CẤU TRÚC DỮ LIỆU (SCHEMA) ===")
df.printSchema()


### Bước 3: Mẫu thông tin trận

- Lấy với cột game: gameId, loại game, thời gian, tên người chơi đầu tiên, thứ hạng.

- Mục đích: kiểm tra nhanh dữ liệu (sanity check) về thời gian và định dạng.


In [ ]:
from pyspark.sql.functions import from_unixtime

df.select(
    col("info.gameId"),
    col("info.tft_game_type"),
    from_unixtime(col("info.game_datetime") / 1000).alias("Ngay_Gio"),
    col("info.participants")[0].getField("riotIdGameName").alias("Ten_Nguoi_Choi_1"),
    col("info.participants")[0].getField("placement").alias("Top_Nguoi_Choi_1")
).show(5, truncate=False)


### Bước 4: Top 5 trận dài nhất

- Sắp xếp game_length giảm dần để xem các outlier.

- Nguyên lý: kiểm tra chất lượng dữ liệu, phát hiện trận bất thường.

In [ ]:
print("=== TOP 5 TRẬN DÀI NHẤT ===")
df.select(
    col("info.gameId").alias("Game_ID"),
    col("info.tft_game_type").alias("Che_Do"),
    col("info.game_length").alias("Thoi_luong_giay")
).orderBy(desc("Thoi_luong_giay")).show(5)


### Bước 5: Top tướng phổ biến

- Flatten participants → units, group by `character_id` đếm số lần xuất hiện.

- Output: top picks (không phụ thuộc vào số liệu cụ thể).

In [ ]:
df_units = df.select(explode(col("info.participants")).alias("player")) \
    .select(explode("player.units").alias("unit"))

df_units.groupBy("unit.character_id") \
    .count() \
    .orderBy(desc("count")) \
    .withColumnRenamed("count", "So_lan_xuat_hien") \
    .show(5, truncate=False)


### Bước 6: Biểu đồ top tướng

- Chuyển Spark DataFrame → Pandas, vẽ bar chart top 10 tướng phổ biến.

- Mục đích: trực quan hóa, có thể thay đổi style/label dễ dàng.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_top_units = df_units.groupBy("unit.character_id").count() \
    .orderBy(desc("count")) \
    .withColumnRenamed("count", "So_lan_xuat_hien") \
    .limit(10)

pdf_units = df_top_units.toPandas()
plt.figure(figsize=(12, 6))
bars = plt.bar(pdf_units['character_id'], pdf_units['So_lan_xuat_hien'], color='#4CAF50', edgecolor='black')
plt.xlabel('Champion', fontsize=12)
plt.ylabel('Pick Count', fontsize=12)
plt.title('Top 10 Most Picked Champions', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.bar_label(bars, fmt='%d', padding=3)
plt.tight_layout()
plt.show()


### Bước 7: Meta tướng + độ phổ biến

- Flatten units kèm `placement`.

- Tính số trận + placement trung bình cho mỗi tướng (lower = tốt).

- Chọn `target_champ` để xem top picks (đếm itemNames).

- Nguyên lý: so sánh theo placement, không cố định số liệu.

In [ ]:
df_players = df.select(
    col("info.gameId"),
    col("info.game_datetime"),
    explode(col("info.participants")).alias("player")
).select(
    col("player.puuid"),
    col("player.placement"),
    col("player.units"),
    col("player.traits")
)

df_units_full = df_players.select("placement", explode("units").alias("unit"))

df_champ_stats = df_units_full.groupBy("unit.character_id") \
    .agg(
        count("*").alias("So_tran_choi"),
        avg("placement").alias("Thu_hang_TB")
    ) \
    .filter(col("So_tran_choi") > 100) \
    .orderBy(col("Thu_hang_TB").asc())

df_champ_stats.show(10)

target_champ = "tft15_leesin"
df_items = df_units_full.filter(col("unit.character_id") == target_champ) \
    .select(explode("unit.itemNames").alias("item_name"))

df_items.groupBy("item_name") \
    .count() \
    .orderBy(desc("count")) \
    .show(5, truncate=False)


### Bước 8: Meta tộc/hệ

- Flatten traits kèm placement.

- Lọc trait được kích hoạt (tier_current > 0).

- Group by name, num_units → tính số trận, placement trung bình.

- Mục đích: xác định mức kích hoạt nào mang lại kết quả tốt.

In [ ]:
df_traits_raw = df_players.select("placement", explode("traits").alias("trait"))
df_trait_stats = df_traits_raw.filter(col("trait.tier_current") > 0) \
    .groupBy("trait.name", "trait.num_units") \
    .agg(
        count("*").alias("So_tran_choi"),
        avg("placement").alias("Thu_hang_TB")
    ) \
    .filter(col("So_tran_choi") > 50) \
    .orderBy(col("Thu_hang_TB").asc())

df_trait_stats.show(10, truncate=False)
